## Exploratory Data Analysis (EDA)
Before diving into modeling, we must understand the data distribution and structure. This phase helps identify the **Class Imbalance** problem and guides our preprocessing strategy.

## Data Summary & Statistics

## TASK 3 — SENTIMENT ANALYSIS: Machine Learning Models            
      Models    : Decision Tree   Naive Bayes                             
      Features  : BoW (3 styles)  GloVe (3 styles)                       
      CV Method : Manual Stratified K-Fold (no data leakage)    
      Methodology: Stratified Manual Cross-Validation | Dynamic Borderline-SMOTE | Feature Importance Selection          

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, BorderlineSMOTE

# Global Configuration
warnings.filterwarnings("ignore")
plt.switch_backend('Agg')

In [3]:
OUTPUT_DIR = Path("Machine_Learning_Outputs")
PLOT_DIR = OUTPUT_DIR / "EDA_Visuals"
REPORT_DIR = OUTPUT_DIR / "Results"
for folder in [PLOT_DIR, REPORT_DIR]: folder.mkdir(parents=True, exist_ok=True)

DATASETS = {
    ("bow", "original"): "bow_original.csv",
    ("bow", "style_b"): "bow_style_b.csv",
    ("bow", "style_c"): "bow_style_c.csv",
    ("glove", "original"): "glove_original.csv",
    ("glove", "style_b"): "glove_style_b.csv",
    ("glove", "style_c"): "glove_style_c.csv",
}

print(f" Workspace ready at: {OUTPUT_DIR}")

 Workspace ready at: Machine_Learning_Outputs


## Exploratory Data Analysis (EDA) Module

## Scientific Exploratory Data Analysis (EDA)
In this section, we analyze the **Feature Matrix** properties before modeling:
* **Sparsity Analysis:** Quantifying information density (crucial for BoW).
* **Label Distribution:** Visualizing the severe class imbalance.
* **Feature Correlation:** Identifying redundancy in high-variance dimensions.

In [4]:
def run_scientific_eda(file_name, rep, style):
    if not Path(file_name).exists(): return None

    df_eda = pd.read_csv(file_name)
    feature_cols = [c for c in df_eda.columns if c not in ["row_id", "ground_truth"]]
    X_eda = df_eda[feature_cols].values
    y_eda = df_eda["ground_truth"].values

    prefix = f"{rep}_{style}"
    sparsity = np.mean(X_eda == 0) * 100

    print(f"\n DATASET PROFILE: {prefix.upper()}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" Sparsity: {sparsity:.2f}% | Features: {X_eda.shape[1]} | Samples: {X_eda.shape[0]}")

    # Visualization
    plt.figure(figsize=(14, 5))

    # Class Balance Plot
    plt.subplot(1, 2, 1)
    sns.countplot(x=y_eda, palette="viridis")
    plt.title(f"Class Distribution ({prefix})")

    # Feature Correlation (Top 10 High-Variance)
    plt.subplot(1, 2, 2)
    top_10 = np.var(X_eda, axis=0).argsort()[-10:]
    sns.heatmap(pd.DataFrame(X_eda[:, top_10]).corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Top 10 Features Correlation")

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"eda_{prefix}.png")
    plt.show()

    return {"rep": rep, "style": style, "sparsity": round(sparsity, 2), "features": X_eda.shape[1]}

# Execute EDA on samples
eda_summaries = []
for (r, s), f in DATASETS.items():
    stats = run_scientific_eda(f, r, s)
    if stats: eda_summaries.append(stats)


 DATASET PROFILE: BOW_ORIGINAL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 99.61% | Features: 10831 | Samples: 442

 DATASET PROFILE: BOW_STYLE_B
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 99.61% | Features: 10783 | Samples: 442

 DATASET PROFILE: BOW_STYLE_C
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 99.61% | Features: 10737 | Samples: 442

 DATASET PROFILE: GLOVE_ORIGINAL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 0.00% | Features: 100 | Samples: 442

 DATASET PROFILE: GLOVE_STYLE_B
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 0.00% | Features: 100 | Samples: 442

 DATASET PROFILE: GLOVE_STYLE_C
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Sparsity: 0.00% | Features: 100 | Samples: 442


🔹 2. Feature Matrix Profiling (Statistical EDA)

Before proceeding to model training, we conducted a rigorous statistical analysis of our feature representations (Bag-of-Words vs. GloVe). This profiling justifies our preprocessing choices, particularly Scaling and Feature Selection.

🔸 Sparsity & Dimensionality Analysis

The table below summarizes the structural differences between the two vectorization methods:

| Dataset        | Feature Type | Total Features | Sparsity (%) | Information Density |
| -------------- | ------------ | -------------- | ------------ | ------------------- |
| BoW_Original   | Discrete     | 10,831         | 99.61%       | Extremely Low       |
| BoW_Style_B    | Discrete     | 10,783         | 99.61%       | Extremely Low       |
| BoW_Style_C    | Discrete     | 10,737         | 99.61%       | Extremely Low       |
| GloVe_Original | Continuous   | 100            | 0.00%        | High                |
| GloVe_Style_B  | Continuous   | 100            | 0.00%        | High                |
| GloVe_Style_C  | Continuous   | 100            | 0.00%        | High                |


###  The Sparsity Challenge and Preprocessing Decisions

#### 1. The Sparsity Challenge (BoW)

- The Bag-of-Words (BoW) representations exhibit **extreme sparsity** (>99.5%). This means that out of approximately 11,000 dimensions, only a tiny fraction of features contains non-zero values for any given sample.

- **Scientific Decision**: This high sparsity justifies the use of `SelectKBest` with the Chi-Square (`chi2`) test to significantly reduce dimensionality by removing noisy and uninformative features, while retaining only the most discriminative ones.

#### 2. Dense vs. Sparse Representations

- While BoW features suffer from massive dimensionality (~11,000 features), GloVe provides a **compact and dense** representation with only 100 features and nearly **0% sparsity**.

- **Scientific Decision**: For GloVe embeddings, we applied `StandardScaler` instead of `MinMaxScaler`. This choice helps preserve the semantic relationships and distances in the continuous vector space, which is critical for embedding-based representations.

#### 3. The "Curse of Dimensionality"

- In the BoW datasets, the number of features (over 11,000) greatly exceeds the number of samples (only 491 instances).

- **Scientific Decision**: To mitigate the risk of overfitting caused by the curse of dimensionality, we implemented:
  - Aggressive feature pruning via `SelectKBest(chi2)`.
  - A high class-weight penalty for the minority (positive) class in the classifiers.

##  Adaptive Sampling & Leakage-Free Validation
We implement a **Manual Stratified Cross-Validation** loop.
- **Sampling:** Adaptive `Borderline-SMOTE` for GloVe and `ROS` for BoW.
- **Zero Leakage:** All preprocessing (Scaling, Selection, Sampling) is fitted **strictly** on training folds.

In [5]:

def get_adaptive_sampler(rep, y_train):
    """
    Selects resampling method based on feature representation type.
    """
    if rep == "bow":
        return RandomOverSampler(random_state=42)
    else:
        # BorderlineSMOTE requires dynamic neighbor adjustment for very small classes
        counts = np.bincount(y_train)
        min_class = np.min(counts[counts > 0])
        k_val = max(1, min(1, min_class - 1)) if min_class > 1 else 1
        return BorderlineSMOTE(k_neighbors=k_val, m_neighbors=max(2, k_val+1), random_state=42)

# Initialize Cross-Validation and results storage
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Primary Model Execution Loop
for (rep, style), fname in DATASETS.items():
    if not Path(fname).exists():
        continue

    # Load and encode data
    df = pd.read_csv(fname)
    feature_cols = [c for c in df.columns if c not in ["row_id", "ground_truth"]]
    X = df[feature_cols].values
    le = LabelEncoder()
    y = le.fit_transform(df["ground_truth"].values)

    # Print Feature Matrix Profile
    print("-" * 60)
    print(f"Dataset Representation: {rep.upper()} | Style: {style}")
    print(f"Matrix Dimensions: {X.shape[0]} samples x {X.shape[1]} features")
    print(f"Class Distribution: {dict(zip(le.classes_, np.bincount(y)))}")
    print("-" * 60)

    for m_name, m_type in [("DecisionTree", "dt"), ("NaiveBayes", "nb")]:
        y_true_agg, y_pred_agg = [], []

        # Stratified K-Fold Cross-Validation
        for tr_idx, ts_idx in cv.split(X, y):
            X_train, X_test = X[tr_idx], X[ts_idx]
            y_train, y_test = y[tr_idx], y[ts_idx]

            # Build Local Pipeline (prevents data leakage)
            steps = []
            if rep == "bow":
                steps.append(('scaler', MinMaxScaler()))
                # Reduce dimensionality for BoW by selecting top 20% features
                steps.append(('select', SelectKBest(chi2, k=int(0.2 * X.shape[1]))))
            else:
                steps.append(('scaler', StandardScaler()))

            # Add adaptive resampling
            steps.append(('sampler', get_adaptive_sampler(rep, y_train)))

            # Classifier setup with minority class weighting
            weights = {0: 1, 1: 1, 2: 50}
            if m_type == "dt":
                clf = DecisionTreeClassifier(class_weight=weights, random_state=42)
            else:
                clf = ComplementNB() if rep == "bow" else GaussianNB()

            steps.append(('clf', clf))
            pipe = ImbPipeline(steps)

            # Training and Prediction
            pipe.fit(X_train, y_train)
            y_pred_agg.extend(pipe.predict(X_test))
            y_true_agg.extend(y_test)

        # Performance Reporting
        y_true_agg, y_pred_agg = np.array(y_true_agg), np.array(y_pred_agg)
        acc = accuracy_score(y_true_agg, y_pred_agg)
        cm = confusion_matrix(y_true_agg, y_pred_agg)

        print(f"Model: {m_name}")
        print(f"Overall Accuracy: {acc:.4f}")
        print("Classification Report:")
        print(classification_report(y_true_agg, y_pred_agg, target_names=le.classes_, digits=4))

        # Confusion Matrix Heatmap Rendering
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f"{m_name} Matrix: {rep}_{style}")
        plt.show()
        plt.close()

        # Save metrics to list
        results.append({
            "Representation": rep, "Style": style, "Model": m_name,
            "Accuracy": round(acc, 4),
            "F1_Macro": round(f1_score(y_true_agg, y_pred_agg, average="macro"), 4),
            "Positive_Recall": round(cm[2,2]/cm[2].sum(), 4) if cm[2].sum() > 0 else 0
        })

# Export finalized results to CSV
final_report = pd.DataFrame(results)
final_report.to_csv(REPORT_DIR / "final_performance_report.csv", index=False)
print("\nModeling cycle complete. Reports saved in directory.")

------------------------------------------------------------
Dataset Representation: BOW | Style: original
Matrix Dimensions: 442 samples x 10831 features
Class Distribution: {'negative': np.int64(190), 'neutral': np.int64(247), 'positive': np.int64(5)}
------------------------------------------------------------
Model: DecisionTree
Overall Accuracy: 0.6131
Classification Report:
              precision    recall  f1-score   support

    negative     0.5797    0.4211    0.4878       190
     neutral     0.6325    0.7733    0.6958       247
    positive     0.0000    0.0000    0.0000         5

    accuracy                         0.6131       442
   macro avg     0.4041    0.3981    0.3945       442
weighted avg     0.6026    0.6131    0.5985       442

Model: NaiveBayes
Overall Accuracy: 0.5905
Classification Report:
              precision    recall  f1-score   support

    negative     0.5424    0.6737    0.6009       190
     neutral     0.7189    0.5385    0.6157       247
    pos

# **2. Scientific Observations**
### **The Imbalance Challenge**

Observation:
Across all models and configurations, the Recall for the Positive class is consistently 0.00.

Reason:
The dataset is extremely imbalanced, containing only 5 positive samples out of 442 instances (~1.1%).

Technical Insight:
Even with the application of:

BorderlineSMOTE
Class-weight adjustments
the minority class remains too underrepresented for the models to learn meaningful patterns. As a result, the models completely fail to generalize for the positive class.

Conclusion:
Unlike the previous experiment, no model was able to detect the positive class at all, indicating that the preprocessing changes did not improve minority class learning.

### **BoW vs. GloVe Dynamics**

**BoW (Bag-of-Words):**

Achieved competitive accuracy, with the best result:
62.90% (Style B + Decision Tree)
Performance relies heavily on keyword frequency, which works well for dominant classes (Negative & Neutral).
However, due to extreme sparsity (~10,000+ features), BoW remains less stable.

**GloVe (Word Embeddings):**

Provided more stable and consistent performance across all styles.
F1-Macro scores are generally higher and more balanced than BoW.
Thanks to:
Dense representation (100 features)
Low sparsity
→ models generalize better.

Key Insight:
GloVe improves balance, but still fails on extreme imbalance cases.

### **Model Comparison: Decision Tree vs. Naive Bayes**

Decision Tree:

Achieves slightly higher accuracy in some BoW cases.
Tends to:
Overfit majority class (Neutral)
Ignore minority class (Positive)
Less stable across representations.

Naive Bayes:

Slightly lower accuracy, but:
More consistent across all setups
Better Negative Recall (often ~67%–78%)
Performs well with both:
Sparse data (BoW)
Dense data (GloVe)


In [6]:
import shutil
from google.colab import files

In [7]:
folder_name = 'Machine_Learning_Outputs'

shutil.make_archive('Outputs', 'zip', folder_name)
files.download('Outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>